# Submit Test – StyleGAN2 1024 ONNX Export & Parameter Check

**목적**: 임의(random) 가중치의 1024 Generator를 ONNX로 내보내고,
PyTorch 파라미터 수 vs ONNX initializer 수를 상세히 비교합니다.

| 단계 | 내용 |
|---|---|
| 1 | 패키지 설치 + 레포 클론 |
| 2 | Google Drive 마운트 (ONNX 저장용) |
| 3 | 랜덤 1024 Generator 생성 |
| 4 | PyTorch 파라미터 상세 분석 |
| 5 | ONNX 내보내기 |
| 6 | ONNX initializer 상세 분석 |
| 7 | PyTorch vs ONNX 최종 비교 |

## 1. 패키지 설치 + 레포 클론

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/jyun-chae/skku-2-openai_pa2.git'
REPO_BRANCH = 'main'
# ─────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

!pip install -q onnx pyyaml

REPO_DIR = '/content/project02'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', REPO_BRANCH], check=True)
    print('Pulled latest.')
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
        check=True,
    )
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print(f'torch={torch.__version__}  CUDA={torch.cuda.is_available()}',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Google Drive 마운트 (ONNX 저장용)

> 로컬 `/content/` 저장만 할 거라면 이 셀은 건너뛰어도 됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── USER CONFIG ───────────────────────────────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/project02'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_DIR, exist_ok=True)
SAVE_TO_DRIVE = True
print(f'Drive mounted. ONNX will be saved to {DRIVE_DIR}')

## 3. 랜덤 1024 Generator 생성

학습된 체크포인트 없이 **랜덤 가중치** 그대로 사용합니다.  
파라미터 수는 가중치 값과 무관하므로 제출 전 count 검증에 충분합니다.

In [ ]:
from src.models.generator import StyleGAN2Generator

# train_1024.yaml 과 동일한 하이퍼파라미터
G = StyleGAN2Generator(
    resolution    = 1024,
    z_dim         = 512,
    w_dim         = 640,
    channel_base  = 65536,
    channel_max   = 512,
    mapping_layers= 8,
    mapping_lr_mul= 0.01,
)
G.eval()

total_pt = G.count_parameters()
print(f'Generator 생성 완료')
print(f'  PyTorch 파라미터 수 : {total_pt:>12,}  ({total_pt/1e6:.3f} M)')
print(f'  40M 제한 여유       : {40_000_000 - total_pt:>12,}  ({(40_000_000 - total_pt)/1e6:.3f} M)')
assert total_pt < 40_000_000, f'PyTorch params {total_pt:,} already exceeds 40M!'

## 4. PyTorch 파라미터 상세 분석

In [ ]:
import pandas as pd

rows = []
for name, param in G.named_parameters():
    rows.append({
        'name'  : name,
        'shape' : list(param.shape),
        'numel' : param.numel(),
        'dtype' : str(param.dtype),
    })

df_pt = pd.DataFrame(rows)
df_pt['numel_M'] = df_pt['numel'] / 1e6

# ── 모듈별 소계 ───────────────────────────────────────────────────────────────
def _module_group(name: str) -> str:
    parts = name.split('.')
    # mapping.net.1.weight  →  mapping
    # b4.conv.weight        →  b4
    # blocks.0.conv1.weight →  blocks.0
    if parts[0] == 'blocks':
        return f'blocks.{parts[1]}'
    return parts[0]

df_pt['group'] = df_pt['name'].apply(_module_group)
summary = (
    df_pt.groupby('group', sort=False)['numel']
    .sum()
    .reset_index()
    .rename(columns={'numel': 'params'})
)
summary['params_M'] = summary['params'] / 1e6
summary['ratio_%']  = summary['params'] / df_pt['numel'].sum() * 100

print('=' * 60)
print('  PyTorch 모듈별 파라미터 수')
print('=' * 60)
print(summary.to_string(index=False))
print('-' * 60)
print(f'  Total  {df_pt["numel"].sum():>12,}  ({df_pt["numel"].sum()/1e6:.3f} M)')
print('=' * 60)

# 전체 파라미터 목록 (접기)
print(f'\n[전체 {len(df_pt)} 개 텐서 목록]')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 60)
display(df_pt[['name', 'shape', 'numel', 'numel_M']].style
        .format({'numel': '{:,}', 'numel_M': '{:.4f}'}))

## 5. ONNX 내보내기

`noise_mode='none'` (zero noise) → 결정론적 그래프로 export.  
constant folding을 켜면 일부 중간 텐서가 initializer로 bake-in되어 ONNX 파라미터 수가 PyTorch보다 늘어납니다.

In [ ]:
import shutil

ONNX_LOCAL = '/content/generator_1024_test.onnx'

onnx_params = G.export_onnx(ONNX_LOCAL, batch_size=1)

# Drive에도 저장
if 'SAVE_TO_DRIVE' in dir() and SAVE_TO_DRIVE:
    dst = f'{DRIVE_DIR}/generator_1024_test.onnx'
    shutil.copy(ONNX_LOCAL, dst)
    print(f'Drive에도 저장됨: {dst}')

## 6. ONNX Initializer 상세 분석

ONNX의 `graph.initializer`는 PyTorch `named_parameters()`와 1:1 대응하지 않습니다.  
constant folding 과정에서 **추가 중간 텐서**가 bake-in 되기 때문입니다.

In [ ]:
import onnx
import numpy as np

model_onnx = onnx.load(ONNX_LOCAL)
initializers = model_onnx.graph.initializer

rows_onnx = []
for init in initializers:
    shape = list(init.dims)
    numel = int(np.prod(shape)) if shape else 1
    dtype_map = {
        1: 'float32', 6: 'int32', 7: 'int64',
        10: 'float16', 11: 'float64',
    }
    dtype = dtype_map.get(init.data_type, f'type{init.data_type}')
    rows_onnx.append({
        'name' : init.name,
        'shape': shape,
        'numel': numel,
        'dtype': dtype,
    })

df_onnx = pd.DataFrame(rows_onnx)
df_onnx['numel_M'] = df_onnx['numel'] / 1e6

total_onnx = df_onnx['numel'].sum()

print('=' * 60)
print(f'  ONNX Initializer 총계')
print('=' * 60)
print(f'  텐서 개수        : {len(df_onnx):,}')
print(f'  총 원소 수       : {total_onnx:,}  ({total_onnx/1e6:.3f} M)')
print(f'  40M 제한 여유    : {40_000_000 - total_onnx:,}  ({(40_000_000 - total_onnx)/1e6:.3f} M)')
status = 'OK ✓' if total_onnx < 40_000_000 else 'OVER LIMIT ✗'
print(f'  상태             : {status}')
print('=' * 60)

# dtype별 분포
print('\n[dtype 별 분포]')
dtype_summary = (
    df_onnx.groupby('dtype')['numel']
    .agg(['count', 'sum'])
    .rename(columns={'count': 'tensors', 'sum': 'total_numel'})
)
dtype_summary['total_M'] = dtype_summary['total_numel'] / 1e6
print(dtype_summary)

# 전체 목록
print(f'\n[전체 {len(df_onnx)} 개 initializer]')
display(df_onnx[['name', 'shape', 'numel', 'dtype', 'numel_M']].style
        .format({'numel': '{:,}', 'numel_M': '{:.4f}'}))

## 7. PyTorch vs ONNX 최종 비교

### 왜 ONNX 파라미터 수가 PyTorch보다 많은가?

`do_constant_folding=True` 옵션으로 export하면 ONNX 런타임이 **상수 식을 미리 계산**하여
`graph.initializer`에 bake-in 합니다. 대표적인 원인:

- `EqualLinear` / `EqualConv2d` 의 `weight * scale` 계산 결과 (학습 후 값이 folded)
- `NoiseInjection`의 `noise_mode='none'` 시 zero 텐서
- ModulatedConv2d 내부 reshape/view 결과물

**제출 기준은 ONNX initializer 수** 이므로, 이 수치가 40M 미만이어야 합니다.

In [ ]:
total_pt   = G.count_parameters()
total_onnx = df_onnx['numel'].sum()
diff       = total_onnx - total_pt
LIMIT      = 40_000_000

print('=' * 65)
print('  최종 비교 요약')
print('=' * 65)
print(f'  PyTorch 파라미터  : {total_pt:>12,}  ({total_pt/1e6:>7.3f} M)')
print(f'  ONNX initializer  : {total_onnx:>12,}  ({total_onnx/1e6:>7.3f} M)')
print(f'  차이 (ONNX - PT)  : {diff:>+12,}  ({diff/1e6:>+7.3f} M)')
print(f'  40M 제한          : {LIMIT:>12,}  ({LIMIT/1e6:>7.3f} M)')
print(f'  PT  여유          : {LIMIT - total_pt:>12,}  ({(LIMIT - total_pt)/1e6:>7.3f} M)')
print(f'  ONNX 여유         : {LIMIT - total_onnx:>12,}  ({(LIMIT - total_onnx)/1e6:>7.3f} M)')
print('-' * 65)
pt_ok   = '✓ PASS' if total_pt   < LIMIT else '✗ FAIL'
onnx_ok = '✓ PASS' if total_onnx < LIMIT else '✗ FAIL'
print(f'  PyTorch 기준 판정 : {pt_ok}')
print(f'  ONNX    기준 판정 : {onnx_ok}   ← 제출 기준')
print('=' * 65)

# ONNX에만 있는 텐서 (constant-folding으로 추가된 것들)
pt_names   = set(name for name, _ in G.named_parameters())
onnx_names = set(df_onnx['name'].tolist())

only_in_onnx = onnx_names - pt_names
only_in_pt   = pt_names   - onnx_names

print(f'\n[ONNX에만 존재하는 initializer: {len(only_in_onnx)} 개]')
if only_in_onnx:
    extra = df_onnx[df_onnx['name'].isin(only_in_onnx)].sort_values('numel', ascending=False)
    print(f'  → constant folding으로 bake-in된 텐서 (총 {extra["numel"].sum():,} 원소)')
    display(extra[['name', 'shape', 'numel', 'dtype']]
            .head(30)
            .style.format({'numel': '{:,}'}))

if only_in_pt:
    print(f'\n[PyTorch에만 존재 (ONNX에서 제거됨): {len(only_in_pt)} 개]')
    print('  (보통 unused params 또는 fused ops)')
    for n in sorted(only_in_pt):
        print(f'    {n}')

## 부록 A. ONNX 구조 검증 (onnx.checker)

StyleGAN2의 `ModulatedConv2d`는 style 벡터로부터 **동적으로 계산된 weight**를 Conv에 넘기므로,
`onnxruntime` CPU/CUDA 구현체는 이를 지원하지 않습니다 (`Conv(11) NOT_IMPLEMENTED`).

onnxruntime 런타임 추론 대신 **onnx.checker** 로 그래프 구조 자체가 유효한지 검증합니다.

In [ ]:
import onnx
from collections import Counter

model_onnx = onnx.load(ONNX_LOCAL)

# 그래프 구조 검증
try:
    onnx.checker.check_model(model_onnx)
    print('onnx.checker : PASS  — 그래프 구조 정상')
except Exception as e:
    print(f'onnx.checker : FAIL  — {e}')

# 그래프 기본 정보
graph = model_onnx.graph
print(f'\n[ONNX 그래프 정보]')
print(f'  opset version : {model_onnx.opset_import[0].version}')
print(f'  inputs        : {[i.name for i in graph.input]}')
print(f'  outputs       : {[o.name for o in graph.output]}')
print(f'  total nodes   : {len(graph.node)}')

# op type 별 분포
op_counts = Counter(n.op_type for n in graph.node)
print(f'\n[Op type 분포]')
for op, cnt in op_counts.most_common():
    print(f'  {op:<25} {cnt:>4}')

print('\n[onnxruntime 추론 불가 이유]')
print('  ModulatedConv2d 는 style 벡터 w 로부터 weight 를 매 forward 마다 동적으로 계산합니다.')
print('  ONNX Conv 스펙은 dynamic weight 를 허용하나,')
print('  onnxruntime CPU/CUDA 구현은 weight 가 static initializer 여야만 동작합니다.')
print('  → 제출 파일 자체는 유효(onnx.checker PASS)하며, 파라미터 수 검증에는 문제 없습니다.')

## 부록 B. PyTorch 추론 테스트 + 샘플 이미지

onnxruntime 대신 **PyTorch 직접 추론**으로 모델이 정상 동작하는지 확인합니다.

In [ ]:
import torch
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G.to(device).eval()

with torch.no_grad():
    z = torch.randn(4, 512, device=device)
    imgs = G(z, noise_mode='const')   # [4, 3, 1024, 1024]

print(f'PyTorch 출력 shape : {imgs.shape}')
print(f'값 범위             : [{imgs.min():.4f}, {imgs.max():.4f}]  (tanh → [-1, 1])')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    img = ((imgs[i].cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'z[{i}]')
plt.suptitle('PyTorch inference — random 1024×1024 (random weights)', y=1.01)
plt.tight_layout()
plt.savefig('/content/pt_test_sample.png', dpi=80, bbox_inches='tight')
plt.show()
print('샘플 이미지 저장: /content/pt_test_sample.png')